# Day 1, Lab 1: Understand the airport data

**Data and AI with Gemini | Corporate Workshop | all data is fictional**

Before we ask an AI anything, we must understand our data. In this notebook you will:

1. Open the airport database with Python
2. See what tables and columns it holds, and what **kind** of data each one is
3. Check the quality of the data: missing values, duplicates, the 4 percent sample
4. Ask your first questions with pandas
5. Build a **simple dashboard**

Run each cell with **Shift + Enter**. Read the text, run the cell, look at the output, then move on.

## Step 1. Set up

In [ ]:
# The tools we need. They are already installed in Google Colab.
import sqlite3                      # lets Python open the database file
import pandas as pd                 # tables (DataFrames) in Python
import matplotlib.pyplot as plt     # charts

print("pandas version:", pd.__version__)

In [ ]:
# Upload the database file. A file picker opens: choose  airport_ops.db
# (it is in the folder 01_Database of the participant pack you unzipped)
from google.colab import files

uploaded = files.upload()
DB = list(uploaded.keys())[0]       # the name of the file you uploaded
print("Using file:", DB)

In [ ]:
# Connect to the database. "mode=ro" means read-only, so nothing can be changed by mistake.
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

def run(sql):
    """Run a SQL query and give the answer back as a table (a pandas DataFrame)."""
    return pd.read_sql_query(sql, con)

print("Connected.")

## Step 2. What is in the database?

A database is a set of **tables**. A table has **rows** (one item each) and **columns** (one fact about the item).
A **view** is a saved question that looks like a table.

In [ ]:
# List every table and view, and count its rows.
tables = run("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")
tables["rows"] = [run(f'SELECT COUNT(*) AS n FROM "{name}"').iloc[0, 0] for name in tables["name"]]
tables

**Your turn (1 minute):** Which table is the biggest? Which has the fewest rows? Which tables sound like *things* (flights, airlines) and which sound like *events* (complaints, incidents)?

## Step 3. Look inside one table

In [ ]:
# Load the flights table into Python and look at the first 5 rows.
flights = run("SELECT * FROM flights")
print("rows and columns:", flights.shape)
flights.head()

## Step 4. The kinds of data

Data comes in three kinds:

| Kind | What it looks like | Example in this workshop |
|---|---|---|
| **Structured** | Rows and columns with fixed types | `flights`, `passenger_journeys`, `staffing_roster` |
| **Semi-structured** | Some structure, some free text | `incidents` (a type, a severity, and a short root cause) |
| **Unstructured** | Free text, no table | The feedback comments and the 12 policy documents (next notebook) |

Inside a table, each column is also a kind of data: an **identifier**, a **category**, a **number you can measure**, a **date**, or **free text**.

In [ ]:
# A helper that describes every column of a table: its type, how many different values, how many are missing.
def column_types(df):
    rows = []
    for col in df.columns:
        values = df[col].dropna()
        unique = values.nunique()
        if pd.api.types.is_numeric_dtype(df[col]):
            kind = "number to measure" if unique > 20 else "number as a code or category"
        elif len(values) > 0 and unique == len(values):
            kind = "identifier"
        elif unique <= 20:
            kind = "category"
        else:
            kind = "text or date"
        example = values.iloc[0] if len(values) else None
        rows.append([col, str(df[col].dtype), unique, int(df[col].isna().sum()), kind, example])
    return pd.DataFrame(rows, columns=["column", "python type", "different values", "missing", "kind of data", "example"])

column_types(flights)

In [ ]:
# The passenger table. This one is a SAMPLE of passengers (see Step 6).
pj = run("SELECT * FROM passenger_journeys")
column_types(pj)

In [ ]:
# Free text: the feedback comments are in English and Arabic.
feedback = run("SELECT * FROM feedback")
feedback[["language", "category", "score", "comment"]].sample(8, random_state=1)

In [ ]:
# Semi-structured: incidents have a type and severity, plus a short free-text root cause.
incidents = run("SELECT * FROM incidents")
incidents[["incident_type", "severity", "duration_min", "root_cause"]].head(6)

## Step 5. Check the quality of the data

In [ ]:
# 1. Missing values: which columns have empty cells?
missing = flights.isna().sum()
print("Flights table, columns with empty cells:")
print(missing[missing > 0])
print()
# delay_reason is only filled for flights that were late by more than 15 minutes
filled = (flights["delay_reason"].fillna("") != "").sum()
print("Flights with a delay reason:", filled, "of", len(flights))

In [ ]:
# 2. Duplicates: the same survey answer submitted twice.
dups = feedback.duplicated(subset=["journey_id", "category", "score", "comment"]).sum()
print("Feedback rows:", len(feedback))
print("Exact duplicate answers:", dups)
print("A clean view exists for this:  v_feedback_clean")
print("Rows in the clean view:", run("SELECT COUNT(*) AS n FROM v_feedback_clean").iloc[0, 0])

In [ ]:
# 3. Time range: which dates does the data cover?
print("First flight date:", flights["flight_date"].min())
print("Last flight date: ", flights["flight_date"].max())
print("Status values:    ", list(flights["status"].unique()))

## Step 6. The 4 percent sample

`passenger_journeys` does not hold every passenger. It holds a random **4 percent sample**.

* To estimate **how many** passengers, multiply the count by **25**.
* **Averages and percentages are never multiplied.**

In [ ]:
sampled = len(pj)
print("Rows in the sample:             ", f"{sampled:,}")
print("Estimated passengers (x 25):    ", f"{sampled * 25:,}")
print()
print("Average spend per passenger:    ", round(pj["spend_aed"].mean(), 1), "AED   <- no scaling for an average")
print("Share who missed their flight:  ", round(pj["missed_flight"].mean() * 100, 2), "%   <- no scaling for a percentage")

## Step 7. Your first questions, answered with pandas

`groupby` means "one answer per group". It is the Python version of SQL's `GROUP BY`.

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Question 1: how many flights per weekday?
flights.groupby("weekday").size().reindex(order)

In [ ]:
# Question 2: which airline has the highest share of delayed flights? (cancelled flights are left out)
operated = flights[flights["status"] != "Cancelled"]
late_share = (operated["status"] == "Delayed").groupby(operated["airline_code"]).mean() * 100
late_share.round(1).sort_values(ascending=False)

In [ ]:
# Question 3: average departure immigration wait per weekday, Terminal 1.
t1_departures = pj[(pj["direction"] == "Departure") & (pj["terminal_id"] == "T1")]
t1_departures.groupby("weekday")["immigration_wait_min"].mean().reindex(order).round(1)

**Look at the result of Question 3.** One weekday stands out. Is it a demand problem (more passengers) or something else? The rest of the workshop is about answering that with evidence.

## Step 8. Your first dashboard

In [ ]:
# Prepare the four small tables we will draw.
flights_per_day = flights.groupby("weekday").size().reindex(order)
wait_per_day = t1_departures.groupby("weekday")["immigration_wait_min"].mean().reindex(order)
scores = run("SELECT category, AVG(score) AS avg_score FROM v_feedback_clean GROUP BY category ORDER BY avg_score")

# One figure with 4 charts (2 rows x 2 columns).
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Airport operations dashboard (fictional data)", fontsize=16, fontweight="bold")

# Chart 1: flights per weekday
ax[0, 0].bar(flights_per_day.index, flights_per_day.values, color="#2b8cbe")
ax[0, 0].set_title("Flights per weekday")
ax[0, 0].tick_params(axis="x", rotation=30)

# Chart 2: delayed share per airline
ax[0, 1].bar(late_share.index, late_share.values, color="#7fb3d5")
ax[0, 1].set_title("Delayed flights per airline (%)")

# Chart 3: average departure wait per weekday (the highest bar in orange)
colors = ["#f39c12" if v == wait_per_day.max() else "#2b8cbe" for v in wait_per_day.values]
ax[1, 0].bar(wait_per_day.index, wait_per_day.values, color=colors)
ax[1, 0].set_title("Terminal 1 departure immigration wait (minutes)")
ax[1, 0].tick_params(axis="x", rotation=30)

# Chart 4: average survey score per category
ax[1, 1].barh(scores["category"], scores["avg_score"], color="#7fb3d5")
ax[1, 1].set_title("Average survey score (1 to 5)")
ax[1, 1].set_xlim(0, 5)

plt.tight_layout()
plt.show()

In [ ]:
# Save the dashboard as a picture and download it.
fig.savefig("dashboard.png", dpi=150, bbox_inches="tight")
files.download("dashboard.png")

## Your turn

Change or add to the dashboard. Pick **one**:

1. Add a chart of **complaints per category** (table `complaints`, column `category`).
2. Add a chart of **average spend per passenger by visa type** (`passenger_journeys`).
3. Add a chart of **incidents per type** (`incidents`, column `incident_type`).
4. Make Chart 3 show **Terminal 2** instead of Terminal 1. What changes?

### What you should be able to say after this notebook
* A database has tables, tables have rows and columns, and every column is a kind of data.
* Some data is structured, some is free text.
* The passenger table is a 4 percent sample: counts x 25, averages never.
* Before using AI on data, we look at it, check it, and draw it.

**Next notebook:** the same understanding, but for the 12 policy documents.